
# Logistic Regression in 2D — Robust Optimizers & Inline Animations

We test GD, L-BFGS, and TR for an ill-conditioned case of Logistic regression.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

rng = np.random.default_rng(42)

def make_ill_conditioned_logreg_data(n=10, noise=0.15, collinear_eps=1e-3, scale_skew=1):
    z = rng.normal(size=n)
    x1_base = z + rng.normal(scale=noise, size=n)
    x2_base = x1_base + rng.normal(scale=collinear_eps, size=n)  # near-collinear
    x1 = scale_skew * x1_base
    x2 = x2_base
    X = np.stack([x1, x2], axis=1)
    w_true = np.array([2.0, -0.5])
    logits = X @ w_true
    p = 1/(1+np.exp(-logits))
    y = (rng.uniform(size=n) < p).astype(float)
    idx = rng.permutation(n)
    return X[idx], y[idx]

X, y = make_ill_conditioned_logreg_data()


In [ ]:

def sigmoid(t):
    t = np.clip(t, -50, 50)
    return 1.0/(1.0+np.exp(-t))

def loss(w, X, y, l2=2e-3):
    z = X @ w
    p = sigmoid(z)
    eps = 1e-12
    nll = -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))
    return nll + 0.5*l2*np.dot(w,w)

def grad(w, X, y, l2=2e-3):
    z = X @ w
    p = sigmoid(z)
    g = X.T @ (p - y) / X.shape[0]
    if l2 > 0:
        g = g + l2*w
    return g

def hess_vec(w, v, X, y, l2=2e-3):
    z = X @ w
    p = sigmoid(z)
    s = (p*(1-p))/X.shape[0]
    Hv = X.T @ (s * (X @ v))
    if l2 > 0:
        Hv = Hv + l2*v
    return Hv

def hessian(w, X, y, l2=2e-3):
    e1 = np.array([1.0, 0.0])
    e2 = np.array([0.0, 1.0])
    return np.column_stack([hess_vec(w, e1, X, y, l2), hess_vec(w, e2, X, y, l2)])


### Strong Wolfe line search

In [ ]:

def strong_wolfe_line_search(f, gfun, w, d, X, y, l2=2e-3, c1=1e-4, c2=0.9, a_max=50.0):
    g0 = gfun(w, X, y, l2)
    if np.dot(g0, d) >= 0:
        d = -g0.copy()
    phi = lambda a: f(w + a*d, X, y, l2)
    dphi = lambda a: np.dot(gfun(w + a*d, X, y, l2), d)

    alpha0 = 0.0
    alpha1 = 1.0
    phi0 = phi(0.0)
    dphi0 = dphi(0.0)

    def zoom(alo, ahi):
        phi_lo = phi(alo)
        while True:
            aj = 0.5*(alo + ahi)
            phi_aj = phi(aj)
            if (phi_aj > phi0 + c1*aj*dphi0) or (phi_aj >= phi_lo):
                ahi = aj
            else:
                dphi_aj = dphi(aj)
                if abs(dphi_aj) <= -c2*dphi0:
                    return aj, phi_aj, gfun(w + aj*d, X, y, l2), d
                if dphi_aj*(ahi-alo) >= 0:
                    ahi = alo
                alo = aj
                phi_lo = phi_aj

    k = 0
    while True:
        phi_a1 = phi(alpha1)
        if (phi_a1 > phi0 + c1*alpha1*dphi0) or (k > 0 and phi_a1 >= phi(alpha0)):
            return zoom(alpha0, alpha1)
        dphi_a1 = dphi(alpha1)
        if abs(dphi_a1) <= -c2*dphi0:
            return alpha1, phi_a1, gfun(w + alpha1*d, X, y, l2), d
        if dphi_a1 >= 0:
            return zoom(alpha1, alpha0)
        alpha0 = alpha1
        alpha1 = min(2.0*alpha1, a_max)
        k += 1


In [ ]:

def newton_solve(w0, X, y, l2=2e-3, iters=60):
    w = w0.copy()
    for _ in range(iters):
        g = grad(w, X, y, l2)
        H = hessian(w, X, y, l2) + 1e-9*np.eye(2)
        try:
            pN = -np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            pN = -g
        a, _, _, d_corr = strong_wolfe_line_search(loss, grad, w, pN, X, y, l2)
        w = w + a*d_corr
        if np.linalg.norm(g) < 1e-8:
            break
    return w

w_opt = newton_solve(np.zeros(2), X, y)
print("Approximate optimum:", w_opt, "loss=", loss(w_opt, X, y))


### Optimizers using strong Wolfe

In [ ]:

def optimize_gd(w0, X, y, l2=2e-3, iters=120):
    traj = [w0.copy()]; vals = [loss(w0, X, y, l2)]
    w = w0.copy()
    for t in range(iters):
        g = grad(w, X, y, l2)
        d = -g
        a, f_new, g_new, d_corr = strong_wolfe_line_search(loss, grad, w, d, X, y, l2)
        w = w + a*d_corr
        traj.append(w.copy()); vals.append(f_new)
    return np.array(traj), np.array(vals)

def optimize_cg_prp(w0, X, y, l2=2e-3, iters=120):
    traj = [w0.copy()]; vals = [loss(w0, X, y, l2)]
    w = w0.copy()
    g_prev = grad(w, X, y, l2)
    d = -g_prev
    for t in range(iters):
        a, f_new, g_new, d_corr = strong_wolfe_line_search(loss, grad, w, d, X, y, l2)
        w_next = w + a*d_corr
        if f_new > vals[-1] - 1e-12 or np.dot(g_new, d_corr) >= 0 or (t+1) % 10 == 0:
            d_next = -g_new
        else:
            yk = g_new - g_prev
            beta_pr = np.dot(g_new, yk)/max(np.dot(g_prev, g_prev), 1e-30)
            beta = max(beta_pr, 0.0)
            d_next = -g_new + beta*d_corr
        w, g_prev = w_next, g_new
        traj.append(w.copy()); vals.append(f_new)
        d = d_next
    return np.array(traj), np.array(vals)

# --- L-BFGS: fast Armijo backtracking + curvature safeguards + early stopping ---

def armijo_backtracking(f, gfun, w, d, X, y, l2=2e-3,
                        alpha0=1.0, c1=1e-4, tau=0.5, max_iter=20):
    """
    Cheap backtracking line search that enforces Armijo (sufficient decrease) only.
    This is typically much faster than Strong Wolfe on ill-conditioned steps.
    """
    g0 = gfun(w, X, y, l2)
    # enforce descent direction
    if np.dot(g0, d) >= 0:
        d = -g0.copy()

    f0 = f(w, X, y, l2)
    g0d = np.dot(g0, d)
    alpha = alpha0

    for _ in range(max_iter):
        w_new = w + alpha*d
        f_new = f(w_new, X, y, l2)
        if f_new <= f0 + c1*alpha*g0d:
            g_new = gfun(w_new, X, y, l2)
            return alpha, f_new, g_new, d
        alpha *= tau

    # last resort very small step
    w_new = w + alpha*d
    return alpha, f(w_new, X, y, l2), gfun(w_new, X, y, l2), d


class LBFGS:
    """
    Limited-memory BFGS with standard two-loop recursion, curvature safeguards,
    and a reset method for robustness.
    """
    def __init__(self, m=10):
        self.m = m
        self.S = []
        self.Y = []
        self.rho = []

    def reset(self):
        self.S.clear(); self.Y.clear(); self.rho.clear()

    def two_loop(self, g):
        # If no history, steepest descent
        if not self.S:
            return -g.copy()

        q = g.copy()
        alphas = []
        for s, y, rho in zip(reversed(self.S), reversed(self.Y), reversed(self.rho)):
            a = rho * np.dot(s, q)
            alphas.append(a)
            q = q - a*y

        # Scale initial Hessian with gamma = s^T y / y^T y (diagonal scaling)
        ys = np.dot(self.Y[-1], self.S[-1])
        yy = np.dot(self.Y[-1], self.Y[-1])
        gamma = ys/yy if yy > 1e-30 else 1.0

        r = gamma*q
        for (s, y, rho), a in zip(zip(self.S, self.Y, self.rho), reversed(alphas)):
            b = rho * np.dot(y, r)
            r = r + s*(a - b)
        return -r

    def update(self, s, y):
        """
        Curvature safeguard: skip update if y^T s is nonpositive or too small.
        """
        ys = np.dot(y, s)
        if ys <= 1e-12 or ys <= 1e-6*np.linalg.norm(s)*np.linalg.norm(y):
            return False
        rho_k = 1.0/ys
        self.S.append(s); self.Y.append(y); self.rho.append(rho_k)
        if len(self.S) > self.m:
            self.S.pop(0); self.Y.pop(0); self.rho.pop(0)
        return True


def optimize_lbfgs_fast(w0, X, y, l2=2e-3, iters=120, m=10,
                        ls_alpha0=1.0, tol_g=1e-6, tol_rel_f=1e-9,
                        verbose=False):
    """
    L-BFGS with:
      - capped Armijo backtracking (no Wolfe zoom stalls)
      - curvature safeguards and memory reset if step becomes non-descent
      - early stopping on grad-norm or tiny relative loss change
    Returns: (trajectory, losses)
    """
    w = w0.copy()
    g = grad(w, X, y, l2)
    f0 = loss(w, X, y, l2)
    traj = [w.copy()]
    vals = [f0]

    lb = LBFGS(m=m)

    for t in range(iters):
        # direction from two-loop (or -g on first step)
        d = lb.two_loop(g)

        # Cheap Armijo backtracking (capped)
        a, f_new, g_new, d_corr = armijo_backtracking(
            loss, grad, w, d, X, y, l2,
            alpha0=ls_alpha0, c1=1e-4, tau=0.5, max_iter=20
        )

        # If step wasn’t descent numerically, reset and take steepest descent
        if np.dot(g, d_corr) >= 0:
            lb.reset()
            d_corr = -g
            a, f_new, g_new, d_corr = armijo_backtracking(
                loss, grad, w, d_corr, X, y, l2,
                alpha0=ls_alpha0, c1=1e-4, tau=0.5, max_iter=20
            )

        # Update iterate and L-BFGS memory
        w_new = w + a*d_corr
        s = w_new - w
        yk = g_new - g
        _ = lb.update(s, yk)  # may skip if curvature poor

        w, g = w_new, g_new
        traj.append(w.copy())
        vals.append(f_new)

        # Early stopping
        if np.linalg.norm(g) < tol_g:
            if verbose:
                print(f"[L-BFGS] stop: ||g||={np.linalg.norm(g):.3e} @ iter {t+1}")
            break
        if abs(vals[-2] - vals[-1]) <= tol_rel_f*max(1.0, abs(vals[-2])):
            if verbose:
                print(f"[L-BFGS] stop: tiny Δf @ iter {t+1}")
            break

        if verbose and (t+1) % 10 == 0:
            print(f"[L-BFGS] it {t+1:03d}  f={vals[-1]:.6f}  ||g||={np.linalg.norm(g):.3e}")

    return np.array(traj), np.array(vals)

In [ ]:

def steihaug_tr_cg(w, g, hvp, X, y, l2, Delta, eta=0.1, max_cg=100):
    z = np.zeros_like(w)
    r = g.copy()
    d = -r
    rTr = np.dot(r, r)
    for _ in range(max_cg):
        Hd = hvp(w, d, X, y, l2)
        dHd = np.dot(d, Hd)
        if dHd <= 0:
            a = np.dot(d, d); b = 2*np.dot(z, d); c = np.dot(z, z) - Delta**2
            tau = (-b + np.sqrt(max(b*b - 4*a*c, 0))) / (2*a)
            return z + tau*d
        alpha = rTr / dHd
        z_next = z + alpha*d
        if np.linalg.norm(z_next) >= Delta:
            a = np.dot(d, d); b = 2*np.dot(z, d); c = np.dot(z, z) - Delta**2
            tau = (-b + np.sqrt(max(b*b - 4*a*c, 0))) / (2*a)
            return z + tau*d
        r_next = r + alpha*Hd
        if np.linalg.norm(r_next) < eta*np.linalg.norm(g):
            return z_next
        beta = np.dot(r_next, r_next)/max(rTr, 1e-30)
        d = -r_next + beta*d
        z, r, rTr = z_next, r_next, np.dot(r_next, r_next)
    return z

def optimize_trust_region(w0, X, y, l2=2e-3, iters=120, Delta0=1.0):
    w = w0.copy()
    traj = [w.copy()]; vals = [loss(w, X, y, l2)]
    Delta = Delta0
    for _ in range(iters):
        g = grad(w, X, y, l2)
        p = steihaug_tr_cg(w, g, hess_vec, X, y, l2, Delta, eta=0.1, max_cg=100)
        Hp = hess_vec(w, p, X, y, l2)
        m0 = loss(w, X, y, l2)
        mp = m0 + np.dot(g, p) + 0.5*np.dot(p, Hp)
        f0 = m0; f1 = loss(w + p, X, y, l2)
        ared = f0 - f1; pred = f0 - mp
        rho = ared / (pred + 1e-30)
        if rho < 0.25:
            Delta *= 0.25
        elif rho > 0.75 and abs(np.linalg.norm(p) - Delta) < 1e-8:
            Delta = min(2.0*Delta, 100.0)
        if rho > 0.0:
            w = w + p
        traj.append(w.copy()); vals.append(loss(w, X, y, l2))
    return np.array(traj), np.array(vals)


In [ ]:

# Run all methods
w_start = np.array([10.0, -10.0])
iters = 120
traj_gd, vals_gd = optimize_gd(w_start, X, y, iters=iters)
# traj_cg, vals_cg = optimize_cg_prp(w_start, X, y, iters=iters)
traj_lb, vals_lb = optimize_lbfgs_fast(w_start, X, y, iters=iters, m=10)
traj_tr, vals_tr = optimize_trust_region(w_start, X, y, iters=iters, Delta0=1.0)

print("Final losses:")
print(" GD      :", vals_gd[-1])
# print(" CG (PR+):", vals_cg[-1])
print(" L-BFGS  :", vals_lb[-1])
print(" TR-CG   :", vals_tr[-1])


In [ ]:

# Build landscape for animations
def build_landscape(trajs, extra_points):
    all_w = np.vstack(trajs + [p[None,:] for p in extra_points])
    w1_min, w1_max = all_w[:,0].min()-2.0, all_w[:,0].max()+2.0
    w2_min, w2_max = all_w[:,1].min()-2.0, all_w[:,1].max()+2.0
    w1s = np.linspace(w1_min, w1_max, 250)
    w2s = np.linspace(w2_min, w2_max, 250)
    W1, W2 = np.meshgrid(w1s, w2s)
    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            Z[i,j] = loss(np.array([W1[i,j], W2[i,j]]), X, y)
    return W1, W2, Z

w_opt = newton_solve(np.zeros(2), X, y)
W1, W2, Z = build_landscape([traj_gd, traj_lb, traj_tr], [w_start, w_opt])


In [ ]:

# Inline animation helper
def animate_inline(traj, title):
    fig, ax = plt.subplots(figsize=(7,6))
    cs = ax.contour(W1, W2, Z, levels=40)
    ax.clabel(cs, inline=1, fontsize=7)
    ax.scatter([w_start[0]], [w_start[1]], marker='x', s=80, label='start')
    ax.scatter([w_opt[0]], [w_opt[1]], marker='*', s=180, label='approx. optimum')
    line, = ax.plot([], [], '-o', ms=3, lw=1.5)
    head, = ax.plot([], [], 'o', ms=6)
    ax.set_title(title)
    ax.set_xlabel("$w_1$"); ax.set_ylabel("$w_2$")
    ax.legend()

    def init():
        line.set_data([], [])
        head.set_data([], [])
        return line, head

    def update(k):
        xs = traj[:k+1,0]; ys = traj[:k+1,1]
        line.set_data(xs, ys)
        head.set_data([xs[-1]], [ys[-1]])
        return line, head

    anim = animation.FuncAnimation(fig, update, init_func=init, frames=len(traj), interval=80, blit=True)
    display(HTML(anim.to_jshtml()))
    plt.close(fig)

# Render animations
animate_inline(traj_gd, "GD + Strong Wolfe")
# animate_inline(traj_cg, "CG (PR+) + Restarts + Strong Wolfe")
animate_inline(traj_lb, "L-BFGS (m=10) + Safeguards + Strong Wolfe")
animate_inline(traj_tr, "Trust Region (Steihaug)")


## Convergence (Loss vs Iteration)

In [ ]:

plt.figure(figsize=(8,5))
plt.plot(vals_gd, label='GD')
# plt.plot(vals_cg, label='CG (PR+)')
plt.plot(vals_lb, label='L-BFGS (m=10)')
plt.plot(vals_tr, label='Trust Region')
plt.xlabel("Iteration"); plt.ylabel("Loss"); plt.title("Convergence vs iteration")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# If you don't already have sigmoid defined:
def sigmoid(t):
    t = np.clip(t, -50, 50)  # numeric safety
    return 1.0/(1.0+np.exp(-t))

def plot_logreg_2d(X, y, w, b=0.0, title="Logistic classifier in 2D",
                   levels=(0.2, 0.5, 0.8), pad=0.5, grid_n=400, alpha_bg=0.20):
    """
    Visualize a 2D logistic regression classifier and data.

    Args
    ----
    X : (n,2) array        Data in 2D.
    y : (n,) array         Labels in {0,1}.
    w : (2,) array         Classifier weights.
    b : float              Intercept (use 0.0 if your model has no bias).
    title : str            Figure title.
    levels : tuple         Probability contour levels to draw.
    pad : float            Padding added around data bounds.
    grid_n : int           Resolution of background grid per axis.
    alpha_bg : float       Background probability colormap alpha.
    """

    # Bounds for the plot
    x1_min, x1_max = X[:,0].min() - pad, X[:,0].max() + pad
    x2_min, x2_max = X[:,1].min() - pad, X[:,1].max() + pad

    # Grid for probabilities
    xs = np.linspace(x1_min, x1_max, grid_n)
    ys = np.linspace(x2_min, x2_max, grid_n)
    XX, YY = np.meshgrid(xs, ys)
    Z = w[0]*XX + w[1]*YY + b
    P = sigmoid(Z)

    # Plot
    fig, ax = plt.subplots(figsize=(7,6))

    # Background probability field
    # Use imshow for speed; origin='lower' to match increasing y upward.
    im = ax.imshow(
        P, extent=[x1_min, x1_max, x2_min, x2_max],
        origin='lower', aspect='auto', alpha=alpha_bg, cmap='RdBu_r',
        vmin=0.0, vmax=1.0
    )

    # Probability contours (including the decision boundary at p=0.5)
    CS = ax.contour(XX, YY, P, levels=levels, colors='k', linewidths=1.25)
    ax.clabel(CS, fmt={lvl: f"p={lvl:.1f}" for lvl in levels}, inline=True, fontsize=9)

    # Data points
    y = np.asarray(y).astype(int)
    mask1 = (y == 1)
    mask0 = ~mask1
    ax.scatter(X[mask0,0], X[mask0,1], s=30, marker='o', edgecolor='k', label='y=0', alpha=0.8)
    ax.scatter(X[mask1,0], X[mask1,1], s=30, marker='^', edgecolor='k', label='y=1', alpha=0.8)

    # Decision boundary line explicitly (p=0.5 -> w·x + b = 0)
    if abs(w[1]) > 1e-12:
        x_line = np.array([x1_min, x1_max])
        y_line = -(w[0]*x_line + b)/w[1]
        ax.plot(x_line, y_line, '--', lw=2, color='k', label='p=0.5 boundary')
    else:
        # Vertical boundary when w[1] ~ 0
        x_vert = -b/w[0]
        ax.axvline(x_vert, ls='--', lw=2, color='k', label='p=0.5 boundary')

    # Cosmetics
    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_title(title)
    ax.legend(loc='best')
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Predicted probability p(y=1|x)")
    plt.tight_layout()
    plt.show()

# ---------------------------
# USAGE EXAMPLES (pick one):
# ---------------------------

# 1) If you used trust region and have its last iterate:
# plot_logreg_2d(X, y, traj_tr[-1], b=0.0, title="TR solution")

# 2) If you used L-BFGS helper and want to plot its final iterate:
# plot_logreg_2d(X, y, traj_lb[-1], b=0.0, title="L-BFGS solution")

# 3) If you computed an approximate optimum via Newton:
plot_logreg_2d(X, y, w_opt, b=0.0, title="Newton (approx. optimum)")

# 4) If your model includes an intercept (bias) term b*, pass it:
# plot_logreg_2d(X, y, w, b=b_star, title="With intercept")